# Train the condensed brain masker on Google Colab

This notebook runs the bundle's training, mask-generation, and evaluation commands.
It mounts Drive, stages the code and dataset onto local runtime storage, updates
`configuration/run_config.json` with fixed local paths, and mirrors resumable state to Drive.

**Run the cells in order.** After a disconnect, run them again from the top with the same
`RUN_NAME` and settings. Training continues from the last mirrored completed epoch.

**Training and inference run in fresh subprocesses.** The notebook executes only the
stdlib configuration and fingerprint helpers in its own kernel. It does not import the
trainer or numerical pipeline, so restaging code does not leave an older version cached
in the notebook. Each new subprocess reads the staged files and installed dependencies.
Keep model and data imports inside those subprocesses.

**Keep local paths fixed.** Resume compares the resolved paths of outputs and all scans
and masks in the split. `CODE_DIR`, `DATA_DIR`, and `RUNS_DIR` therefore stay at the same
local paths across sessions; Drive holds backups.


## 1. Settings - edit this cell only

The Colab defaults use **128 x 128 x 128 training crops**, **one validation window at a time**, and **two data workers** to reduce memory use on a 16 GB GPU. The full scan is still conformed to 256 x 256 x 256 at 1 mm; validation and test Dice cover full scans assembled from overlapping windows. Crops preserve voxel resolution but give the model less spatial context per training example.

`EPOCHS=None` uses `configuration/run_config.json`. `PATCH_SIZE=None` selects whole-volume training for a larger GPU. Model, loss, augmentation, and split settings come from `configuration/run_config.json`. Choose a new `RUN_NAME` when changing training settings; `INIT_FROM` keeps the previous model's learned weights.


In [ ]:
RUN_NAME = "condensed_masker_all_augs_128patch_v4"   # a NEW name whenever any setting changes

# --- Sources on Drive --------------------------------------------------------
# CODE_SRC / DATA_SRC may each be a .zip or an already-unpacked folder.
# Prefer zips: Drive reads one archive far faster than thousands of small files.
CODE_SRC  = "/content/drive/MyDrive/deployment_condensed.zip"
DATA_SRC  = "/content/drive/MyDrive/NFBS_Dataset.zip"
DRIVE_DIR = "/content/drive/MyDrive/condensed_runs"   # where resumable state is mirrored

# --- Colab overrides. Choose BEFORE the first launch; keep fixed on resume ---
EPOCHS       = None    # e.g. 60. FROZEN into resume identity by the first launch - see cell 6.
NUM_WORKERS  = 2       # this runtime reports 2 CPUs; also frozen into resume identity
PATCH_SIZE = (128, 128, 128)  # 1 mm training crops; None = whole-volume training
SW_BATCH_SIZE = 1           # validation/test windows forwarded together (limits GPU memory)
BACKUP_EVERY_MIN = 10  # how often training state is mirrored to Drive (see cell 9 - it is ~1.5 GB)
INIT_FROM = '/content/drive/MyDrive/condensed_runs/condensed_masker_minimal_augs/mask.pt'  # starting weights for a NEW run; saved training state takes precedence

# --- Fixed LOCAL paths. Never change these between sessions of one run -------
CODE_DIR = "/content/deployment_condensed"
DATA_DIR = "/content/data"
RUNS_DIR = "/content/runs"

RUN_DIR       = f"{RUNS_DIR}/{RUN_NAME}"
DRIVE_RUN_DIR = f"{DRIVE_DIR}/{RUN_NAME}"
print("run:", RUN_NAME, "\nlocal:", RUN_DIR, "\ndrive:", DRIVE_RUN_DIR)

## 2. Mount Drive

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

# Catch the easy mistake before anything is copied.
for _name, _p in (('CODE_DIR', CODE_DIR), ('DATA_DIR', DATA_DIR), ('RUNS_DIR', RUNS_DIR)):
    assert not _p.startswith('/content/drive'), (
        f'{_name}={_p} is on Drive. Training paths must be local /content/... - see cell 1.')
for _name, _p in (('CODE_SRC', CODE_SRC), ('DATA_SRC', DATA_SRC)):
    assert os.path.exists(_p), f'{_name}={_p} does not exist. Check the path in cell 1.'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('drive mounted; sources found')

## 3. Stage the code

Wipes `CODE_DIR` and re-extracts, so a file the new zip no longer contains cannot linger, and no
stale `__pycache__` survives. A zip that carries its own top-level folder is flattened, rather
than leaving two copies of `train.py` on disk.

The **source fingerprint** printed here is the same hash the trainer embeds in its checkpoint
(the training, model, imaging, configuration, and augmentation sources).
`training/provenance.py` defines the file list used by both the notebook and trainer.
It must be identical in every session of one run — a different value means the zip changed and
the checkpoint will refuse to resume.
Staging prints the exact source ZIP, its SHA-256, and the selected internal folder. Flat ZIPs and nested wrapper folders are supported. Multiple complete code copies are rejected rather than choosing one silently. Augmentation-evaluation helpers are optional for training and are checked when the evaluation cell runs.


In [ ]:
import datetime, hashlib, os, pathlib, shutil, tempfile, zipfile


def code_fingerprint(root):
    """Use the trainer's stdlib-only fingerprint implementation before numerical imports."""
    import runpy
    root = pathlib.Path(root)
    provenance = runpy.run_path(str(root / 'training/provenance.py'))
    return provenance['training_code_fingerprint'](root)


# Training does not depend on the optional augmentation-evaluation helpers.
_TRAINING_FILES = (
    'train.py',
    'training/__init__.py',
    'training/engine.py',
    'training/config.py',
    'training/data.py',
    'training/checkpoints.py',
    'training/runtime.py',
    'training/provenance.py',
    'imaging/__init__.py',
    'imaging/geometry.py', 'imaging/normalization.py', 'imaging/metadata.py',
    'models/__init__.py',
    'models/architectures.py',
    'models/losses.py',
    'configuration/__init__.py',
    'configuration/presets.py',
    'configuration/augmentation_presets/standard.json',
    'configuration/run_config.json',
    'requirements.txt',
)
_EVALUATION_FILES = (
    'generate_mask.py',
    'evaluate.py',
    'evaluate_augmentations.py',
    'inference/__init__.py',
    'inference/masker.py',
    'inference/checkpoints.py',
    'evaluation/__init__.py',
    'evaluation/standard.py',
    'evaluation/augmentation.py',
    'evaluation/transforms.py',
    'configuration/run_config.augmented.json',
)


def stage_code(source, destination):
    source = pathlib.Path(source).expanduser().resolve()
    destination = pathlib.Path(destination).expanduser().resolve()
    if not source.exists():
        raise FileNotFoundError(f'CODE_SRC does not exist: {source}')
    if (destination == pathlib.Path(destination.anchor) or source == destination
            or destination in source.parents
            or (source.is_dir() and source in destination.parents)):
        raise ValueError('CODE_SRC and CODE_DIR must be separate, non-overlapping locations; CODE_DIR cannot be a filesystem root.')
    print('Reading CODE_SRC:', source)
    if source.is_file():
        st = source.stat()
        with source.open('rb') as stream:
            archive_hash = hashlib.file_digest(stream, 'sha256').hexdigest()
        print(f'Archive: {st.st_size:,} bytes | modified UTC {datetime.datetime.fromtimestamp(st.st_mtime, datetime.timezone.utc).isoformat()}')
        print('Archive SHA-256:', archive_hash)
        nearby = sorted(path.name for path in source.parent.glob('deployment_condensed*.zip')
                        if path != source)
        if nearby:
            print('Other nearby ZIPs (not selected):', nearby)

    # Validate in a temporary directory before replacing an existing code copy.
    # Recursive root discovery supports flat ZIPs and any number of wrappers.
    with tempfile.TemporaryDirectory(prefix='masker-code-staging-') as scratch:
        unpacked = pathlib.Path(scratch) / 'unpacked'
        if source.is_dir():
            shutil.copytree(source, unpacked, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
        else:
            unpacked.mkdir()
            with zipfile.ZipFile(source) as archive:
                archive.extractall(unpacked)
        candidates = sorted({path.parent for path in unpacked.rglob('train.py')
                             if '__MACOSX' not in path.parts
                             and all((path.parent / name).is_file() for name in _TRAINING_FILES)
                             and (path.parent / 'augmentations').is_dir()})
        if len(candidates) != 1:
            train_locations = [path.relative_to(unpacked).as_posix()
                               for path in unpacked.rglob('train.py')][:20]
            nested_archives = [path.relative_to(unpacked).as_posix()
                               for path in unpacked.rglob('*.zip')][:20]
            raise ValueError(
                f'Expected one training bundle inside {source}; found {len(candidates)}. '
                f'Candidate roots: {[path.relative_to(unpacked).as_posix() for path in candidates]}. '
                f'train.py locations: {train_locations}; nested ZIPs: {nested_archives}. '
                'Check CODE_SRC selects the intended archive, containing the Python files and augmentations folder. '
                'The existing CODE_DIR was left untouched.')
        selected = candidates[0]
        print('Selected bundle root:', selected.relative_to(unpacked).as_posix())
        # The resolved destination is exactly the named CODE_DIR, checked above
        # not to overlap the source or a filesystem root.
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(selected, destination,
                        ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))

    missing = [name for name in _EVALUATION_FILES if not (destination / name).is_file()]
    if missing:
        print('Training code is ready. Optional evaluation files missing:', ', '.join(missing))
        print('To use evaluation, select the updated bundle ZIP or restore the missing files at the listed paths. '
              'Training can continue without them.')
    else:
        print('Training and augmentation-evaluation files are present.')
    return destination


stage_code(CODE_SRC, CODE_DIR)

CODE_FINGERPRINT = code_fingerprint(CODE_DIR)
print('code staged at', CODE_DIR)
print('source fingerprint:', CODE_FINGERPRINT[:16],
      '\n  (must be identical in every session of this run)')

## 4. Install dependencies and check the runtime

The runtime report runs in a **subprocess**, so it shows the versions the trainer will actually
record — not whatever this kernel happens to have imported. Those versions are part of resume
identity: if Colab ships a new torch between sessions, an existing run cannot resume and must be
warm-started from its weights instead (`--init-from`).

In [ ]:
!pip install -q -r {CODE_DIR}/requirements.txt

import os, subprocess, sys

_probe = r"""
import json, sys
import numpy, torch, nibabel
from importlib import metadata
def dist(n):
    try: return metadata.version(n)
    except metadata.PackageNotFoundError: return None
print(json.dumps({
    'python': sys.version.split()[0], 'torch': str(torch.__version__),
    'numpy': str(numpy.__version__), 'nibabel': str(nibabel.__version__),
    'cuda_runtime': str(torch.version.cuda), 'scipy': dist('scipy'), 'monai': dist('monai'),
    'torchvision': dist('torchvision'), 'opencv_python': dist('opencv-python'),
    'opencv_python_headless': dist('opencv-python-headless'),
    'scikit_image': dist('scikit-image'),
    'cuda_available': torch.cuda.is_available(),
    'gpu': (torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
}, indent=1))
"""
_out = subprocess.run([sys.executable, '-c', _probe], cwd=CODE_DIR, text=True,
                      capture_output=True)
print(_out.stdout or _out.stderr)
print('logical CPUs:', os.cpu_count())
if '"cuda_available": true' not in _out.stdout.lower():
    print('\n! No GPU. Runtime > Change runtime type > GPU, then re-run from cell 2.')

## 5. Stage the dataset onto local disk

**Not optional.** The trainer hashes every scan and mask and then re-stats each file to prove it
did not change mid-read; Drive's FUSE driver bumps the timestamps when it materialises a file for
that read, so the tamper check fires on the very first scan. Online synthesis also re-reads every
scan and mask once per sample per epoch, which over FUSE would dominate the run.

Staging is idempotent — re-running it in a later session reuses what is already there.

`data_dir` is derived as the common parent of every discovered scan, so an archive that wraps the
dataset in one or more folders still resolves to the same place. **Use the same `DATA_SRC` every
session**: a zip and a loose folder of the same data can land at different absolute paths, and
those paths are baked into the split fingerprint.

Read the two lines this prints. `data_dir` and the scan-path digest are the user-visible proxies
for the fingerprint the trainer compares on resume, and the first pair names the mask file
actually used — a `mask_suffix` that matches nothing fails loudly here rather than at epoch 1.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, zipfile

_cfg_src = json.loads(pathlib.Path(CODE_DIR, 'configuration/run_config.json').read_text(encoding='utf-8'))
SCAN_GLOB   = _cfg_src['training']['scan_glob']
MASK_SUFFIX = _cfg_src['training']['mask_suffix']


def _scans(root):
    return sorted(p for p in pathlib.Path(root).rglob(SCAN_GLOB)
                  if 'brainmask' not in p.name and '_brain.' not in p.name)


if _scans(DATA_DIR):
    print('dataset already staged at', DATA_DIR, '- reusing it')
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.isdir(DATA_SRC):
        # Copy INTO a subfolder named after the source, so a folder and a zip of the same
        # dataset produce the same layout.
        shutil.copytree(DATA_SRC, os.path.join(DATA_DIR, os.path.basename(DATA_SRC.rstrip('/'))),
                        dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(DATA_SRC) as zf:
            zf.extractall(DATA_DIR)
    print('staged', DATA_SRC, '->', DATA_DIR)

_found = _scans(DATA_DIR)
assert _found, (
    f'no files matching {SCAN_GLOB!r} under {DATA_DIR}. Check DATA_SRC and the scan_glob in '
    'configuration/run_config.json.')
DATA_ROOT = os.path.commonpath([str(p.parent) for p in _found])

_digest = hashlib.sha256()
for _p in _found:
    _digest.update(str(_p.resolve()).encode('utf-8') + b'\0')
print(f'\n{len(_found)} scans | data_dir = {DATA_ROOT}')
print('scan-path digest:', _digest.hexdigest()[:16],
      '\n  (must be identical in every session of this run)')

# Ground truth for pairing: the trainer's own discovery, run in a subprocess.
_pair = r"""
import json, sys
from augmentations.pipeline import discover_pairs
p = discover_pairs(sys.argv[1], sys.argv[2], sys.argv[3])
print(json.dumps({'n': len(p), 'first': (p[0] if p else None)}, indent=1))
"""
_out = subprocess.run([sys.executable, '-c', _pair, DATA_ROOT, SCAN_GLOB, MASK_SUFFIX],
                      cwd=CODE_DIR, text=True, capture_output=True)
if _out.returncode:
    raise RuntimeError('pair discovery failed - fix mask_suffix in configuration/run_config.json, or the '
                       'dataset layout:\n' + (_out.stderr or _out.stdout))
print('\n[pairing]', _out.stdout)

## 6. Write the run configuration

Patches the staged `configuration/run_config.json` with local paths and the explicit overrides from the settings cell. The Colab crop profile keeps the 256-cube, 1 mm scan geometry and uses 128-cube training crops plus one sliding-window prediction at a time for full-volume validation/test.

**Training settings are frozen by the first launch.** Resume compares the configuration, including planned epochs, workers, crop size, and validation window batch size. Changing these requires a new `RUN_NAME` and a weights-only warm start via `INIT_FROM`.

Workers control independent augmentation RNG streams as well as CPU use. Two workers match the small runtime reported in the failed run; a different count must be chosen before starting another run.

The cell re-checks the source fingerprint afterwards. `configuration/run_config.json` is outside the source hash, but its resolved training settings are checked separately during resume.

The supplied configuration enables the **full standard augmentation mix: benign families plus nonbenign artifacts**. Sampling is 18% clean, 22% canonical MPRAGE/MP2RAGE/T2/FLAIR (about 5.5% each), 30% broad acquisition/appearance variation, and 30% label-driven synthetic anatomy. Broad acquisitions cover all 10 configured sequences, field strengths, vendors, and reconstruction styles. Morphology, orientation/resize, and resolution augment eligible branches. The full standard artifact pool is enabled: all 36 supported 3D overlay transforms. With `p_artifact=0.30`, eligible broad-benign/synthetic draws receive one or two sampled artifacts (about 18% of all training draws reach this gate). Unsupported transforms and optional adversarial, dedicated MP2RAGE stress, and hard-artifact-tail curricula remain off. Protocol examples retain canonical geometry.


In [ ]:
import json, os, pathlib

_cfg_path = pathlib.Path(CODE_DIR, 'configuration/run_config.json')
cfg = json.loads(_cfg_path.read_text(encoding='utf-8'))
_t = cfg['training']

_before = {k: _t.get(k) for k in ('data_dir', 'model_out_path', 'results_out_path',
                                  'epochs', 'num_workers', 'patch_size', 'sw_batch_size')}
_t['data_dir']         = DATA_ROOT
_t['model_out_path']   = f'{RUN_DIR}/mask.pt'
_t['results_out_path'] = f'{RUN_DIR}/mask.json'
if EPOCHS is not None:
    _t['epochs'] = int(EPOCHS)
if NUM_WORKERS is not None:
    _t['num_workers'] = int(NUM_WORKERS)
_t['patch_size'] = list(PATCH_SIZE) if PATCH_SIZE is not None else None
_t['sw_batch_size'] = int(SW_BATCH_SIZE)
_cfg_path.write_text(json.dumps(cfg, indent=2), encoding='utf-8')

for _k, _old in _before.items():
    _new = _t.get(_k)
    print(f'  {_k:18} {_old!r}' + ('' if _new == _old else f'  ->  {_new!r}'))

_cpus = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 1)
if _t['num_workers'] > _cpus:
    print(f'\n! num_workers={_t["num_workers"]} exceeds available CPUs={_cpus}; expect '
          'contention. It is frozen into resume identity, so choose it now.')
if _t['num_workers'] == 0:
    print('\n! num_workers=0: augmentation runs in the training process and will bottleneck the '
          'GPU.\n  Set NUM_WORKERS in cell 1 BEFORE the first launch - it cannot change later.')

assert code_fingerprint(CODE_DIR) == CODE_FINGERPRINT, 'source fingerprint moved unexpectedly'
print('\nsource fingerprint unchanged:', CODE_FINGERPRINT[:16])

import runpy
cfg = runpy.run_path(str(pathlib.Path(CODE_DIR, 'configuration/presets.py')))['resolve_run_config'](_cfg_path)

_enabled_augs = [item['name'] for item in cfg['augmentations'] if item['enabled']]
print('Enabled training augmentation stages:', ', '.join(_enabled_augs))
_protocol_stage = next(item for item in cfg['augmentations'] if item['name'] == 'standard_protocols')
if _protocol_stage['enabled']:
    _protocol_names = [item['name'] for item in _protocol_stage['settings']['protocols']]
    print('Standard protocols:', ', '.join(_protocol_names),
          '| total training share:', cfg['sampling']['p_standard'])


## 7. Restore a previous session's state from Drive

Copies the mirrored run back to its fixed local path. If a *local* run directory is already
further along than the Drive mirror — a disconnect can interrupt a backup — the local copy is
kept and the cell says so. Nothing is overwritten silently.

On a first launch there is nothing to restore and this cell is a no-op.

In [ ]:
import json, os, pathlib, shutil


def _epochs_done(run_dir):
    """Epochs recorded in a run directory's results sidecar, or -1 if there is none."""
    try:
        return int(json.loads(pathlib.Path(run_dir, 'mask.json').read_text(
            encoding='utf-8')).get('epochs_done', -1))
    except (OSError, ValueError, KeyError):
        return -1


os.makedirs(RUN_DIR, exist_ok=True)
_local, _remote = _epochs_done(RUN_DIR), _epochs_done(DRIVE_RUN_DIR)
_have_remote = os.path.isfile(os.path.join(DRIVE_RUN_DIR, 'mask.train_state.pt'))

if not _have_remote:
    print(f'no Drive mirror at {DRIVE_RUN_DIR} - starting fresh'
          if _local < 0 else
          f'no Drive mirror yet; keeping the local run at epoch {_local}')
elif _local > _remote:
    print(f'! KEEPING the local run (epoch {_local}) - the Drive mirror is behind at epoch '
          f'{_remote}.\n  The last backup was probably interrupted. Restoring would lose '
          'committed epochs.')
else:
    for _src in sorted(pathlib.Path(DRIVE_RUN_DIR).glob('*')):
        if _src.is_file() and not _src.name.endswith(('.tmp', '.part')):
            shutil.copyfile(_src, pathlib.Path(RUN_DIR, _src.name))
    print(f'restored {DRIVE_RUN_DIR} -> {RUN_DIR} (epoch {_remote})')

print('\nlocal run directory:')
for _p in sorted(pathlib.Path(RUN_DIR).glob('*')):
    print(f'  {_p.name:28} {_p.stat().st_size / 1e6:9.1f} MB')

## 8. Validate the configuration

Checks every setting and their cross-field requirements without loading the dataset or building a
model. Fix anything it reports before training — the same failures cost an epoch of setup later.

In [ ]:
import json, subprocess, sys

_out = subprocess.run([sys.executable, 'train.py', '--config', 'configuration/run_config.json',
                       '--validate-config'], cwd=CODE_DIR, text=True, capture_output=True)
if _out.returncode:
    raise RuntimeError('configuration is invalid:\n' + (_out.stderr or _out.stdout))

_resolved = json.loads(_out.stdout)
for _k in ('model_type', 'data_dir', 'model_out_path', 'results_out_path', 'epochs',
           'target_shape', 'conform_mm', 'patch_size', 'sw_batch_size', 'amp', 'batch_size', 'accum_steps', 'num_workers', 'lr',
           'val_fraction', 'test_fraction', 'seed', 'model_kwargs', 'loss_kwargs', 'ema',
           'warmup_epochs', 'scan_glob', 'mask_suffix'):
    print(f'  {_k:18} {_resolved.get(_k)!r}')
print('\nconfiguration is valid.')

## 9. Train — and re-run this exact cell after any disconnect

The trainer runs as a subprocess and its output streams here live. A background thread mirrors the
run directory to Drive every `BACKUP_EVERY_MIN` minutes and once more when the cell ends, however
it ends — including when you interrupt it.

`--resume` is added automatically when this run already has a training state, so re-running the
cell continues from the last completed epoch instead of starting over. Re-running a *finished*
run is safe: the trainer reports that it is already complete and exits.

Both the trainer's own writes and the Drive mirror publish through a temporary file and a rename,
so a copy interrupted at any moment leaves the previous complete file in place rather than a
half-written one.

**The backup is not small.** `mask.train_state.pt` carries the model, the EMA shadow, the
optimizer moments and the selected-best snapshot — around 1.5 GB for this configuration, plus
~290 MB for `mask.pt`. Files unchanged since the previous backup are skipped (`mask.pt` only moves
on a new best epoch), but every backup still pushes the state file. If a backup takes longer than
`BACKUP_EVERY_MIN`, raise the interval — the printed MB and seconds tell you.
For a warm start, set `INIT_FROM` in the settings cell to a previous compatible `mask.pt`, and choose a new `RUN_NAME`. Existing training state takes precedence, so reconnecting resumes the new run instead of loading the old weights again. This cell reloads `cfg` from the saved `configuration/run_config.json`; it does not require the configuration variable to survive in kernel memory. Run section 6 first when setting up a new run or newly staging code.


In [ ]:
import json, os, pathlib, shutil, subprocess, sys, threading, time

# This cell can be rerun after a kernel restart: reload the saved run settings
# instead of relying on the in-memory `cfg` created by section 6.
_required = ('CODE_DIR', 'RUN_DIR', 'DRIVE_RUN_DIR', 'BACKUP_EVERY_MIN')
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError('Run the notebook settings cell first. Missing: ' + ', '.join(_missing))
_cfg_path = pathlib.Path(CODE_DIR, 'configuration/run_config.json').resolve()
if not _cfg_path.is_file():
    raise FileNotFoundError(f'{_cfg_path} is missing. Run the code-staging and run-configuration cells first.')
import runpy
cfg = runpy.run_path(str(pathlib.Path(CODE_DIR, 'configuration/presets.py')))['resolve_run_config'](_cfg_path)

# Catch a staged-but-unconfigured bundle before training writes outside the
# directory mirrored to Drive. Section 6 assigns these paths for this run.
for _key, _filename in (('model_out_path', 'mask.pt'), ('results_out_path', 'mask.json')):
    _configured = pathlib.Path(cfg.get('training', {}).get(_key) or '')
    if not _configured.is_absolute():
        _configured = _cfg_path.parent / _configured
    _expected = pathlib.Path(RUN_DIR, _filename).resolve()
    if _configured.resolve() != _expected:
        raise RuntimeError(f'Run section 6 (Write the run configuration) first: {_key} must point to {_expected}.')

_state = pathlib.Path(RUN_DIR, 'mask.train_state.pt')
_cmd = [sys.executable, '-u', 'train.py', '--config', str(_cfg_path)]
_init_from = globals().get('INIT_FROM')
if _state.is_file():
    _cmd += ['--resume', str(_state)]
    print('resuming from', _state)
elif _init_from:
    _init_path = pathlib.Path(_init_from).expanduser()
    if not _init_path.is_absolute():
        _init_path = pathlib.Path(CODE_DIR) / _init_path
    _init_path = _init_path.resolve()
    if not _init_path.is_file():
        raise FileNotFoundError(f'INIT_FROM checkpoint not found: {_init_path}')
    _cmd += ['--init-from', str(_init_path)]
    print('starting a new run from previous model weights:', _init_path)
else:
    print('starting a new run from fresh weights')

# The hard artifact tail demands thread caps established BEFORE any numeric library is imported.
# In a subprocess that is simply the child's environment - no bootstrap gymnastics needed.
_env = dict(os.environ)
if float((cfg.get('sampling') or {}).get('hard_artifact_tail_fraction') or 0.0) > 0.0:
    _env['MASKER_HARDTAIL_THREAD_LIMITS'] = '1'
    print('hard artifact tail enabled -> MASKER_HARDTAIL_THREAD_LIMITS=1 '
          '(needs num_workers=4 and >=16 logical CPUs)')


_mirrored = {}   # name -> (size, mtime_ns) of what this session last pushed to Drive


def backup_to_drive(quiet=True):
    """Mirror the run directory to Drive. Publishes each file atomically; skips in-progress
    writes and files unchanged since this session's previous backup."""
    os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
    copied, sent = 0, 0
    t0 = time.time()
    for src in sorted(pathlib.Path(RUN_DIR).glob('*')):
        if not src.is_file() or src.name.endswith(('.tmp', '.part')):
            continue
        st = src.stat()
        # mask.pt only changes on a new best epoch; skipping it saves ~290 MB per backup.
        if _mirrored.get(src.name) == (st.st_size, st.st_mtime_ns):
            continue
        dst, part = pathlib.Path(DRIVE_RUN_DIR, src.name), pathlib.Path(DRIVE_RUN_DIR, src.name + '.part')
        try:
            shutil.copyfile(src, part)
            os.replace(part, dst)
        except OSError:
            part.unlink(missing_ok=True)
            shutil.copyfile(src, dst)   # some FUSE mounts refuse rename; direct copy is the fallback
        _mirrored[src.name] = (st.st_size, st.st_mtime_ns)
        copied, sent = copied + 1, sent + st.st_size
    if not quiet:
        print(f'[backup] {copied} file(s), {sent / 1e6:.0f} MB in {time.time() - t0:.0f}s '
              f'-> {DRIVE_RUN_DIR}  {time.strftime("%H:%M:%S")}')
    return copied


_stop = threading.Event()


def _mirror():
    while not _stop.wait(BACKUP_EVERY_MIN * 60):
        try:
            backup_to_drive(quiet=False)
        except Exception as exc:            # a Drive hiccup must never kill the training run
            print('[backup] FAILED:', exc)


_thread = threading.Thread(target=_mirror, daemon=True)
_thread.start()
_proc = subprocess.Popen(_cmd, cwd=CODE_DIR, env=_env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for _line in _proc.stdout:
        print(_line, end='')
    _rc = _proc.wait()
finally:
    _stop.set()
    _proc.poll() is None and _proc.terminate()
    try:
        _proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        _proc.kill()
    print('\n[backup] final mirror to Drive ...')
    backup_to_drive(quiet=False)

print('\ntrain.py exit code:', _proc.returncode)
if _proc.returncode:
    print('Training did not finish. Fix the error above, then re-run this cell - the run '
          'continues from its last completed epoch.')

## 10. Inspect progress

`selection_metric` names the number that actually chose the deployed checkpoint.
`training_complete` is `false` on every intermediate write and only becomes `true` after the run
finishes its planned epochs — an intermediate checkpoint can be evaluated diagnostically, but is not a completed training run. Keep final test evaluation separate from model tuning.

In [ ]:
import json, pathlib

_res = json.loads(pathlib.Path(RUN_DIR, 'mask.json').read_text(encoding='utf-8'))
print('complete:', _res['training_complete'],
      '| epochs', _res['epochs_done'], '/', _res['epochs_total'])
print('selection metric:', _res['selection_metric'],
      '| best', _res['best_selection_score'], 'at epoch', _res['best_epoch'])
print('\nlast 10 epochs:')
for _row in _res['history'][-10:]:
    print(' ', _row)

## 11. Backup status — check this before ending the session

Colab can stop a runtime without warning. Anything after the most recent complete backup is lost,
so confirm the Drive mirror is level with the local run before you close the tab.

In [ ]:
import os, pathlib, time

print(f'local {RUN_DIR}: epoch {_epochs_done(RUN_DIR)}')
print(f'drive {DRIVE_RUN_DIR}: epoch {_epochs_done(DRIVE_RUN_DIR)}')
for _p in sorted(pathlib.Path(DRIVE_RUN_DIR).glob('*')) if os.path.isdir(DRIVE_RUN_DIR) else []:
    print(f'  {_p.name:28} {_p.stat().st_size / 1e6:9.1f} MB  '
          f'{time.strftime("%Y-%m-%d %H:%M", time.localtime(_p.stat().st_mtime))}')

if _epochs_done(RUN_DIR) > _epochs_done(DRIVE_RUN_DIR):
    print('\n! Drive is BEHIND. Run the next line, then check again.')
    print('  backup_to_drive(quiet=False)')
else:
    print('\nDrive mirror is current - safe to end the session.')

## 12. Use the trained model

`mask.pt` is the selected checkpoint and embeds its own architecture and preprocessing settings —
it is the only file inference needs. Both commands below are also how you would run them on your
own machine; only the paths change.

Uncomment and point them at a real scan.

In [ ]:
!ls -lh {RUN_DIR}

# Generate a native-space binary mask for one scan:
# !cd {CODE_DIR} && python generate_mask.py \
#     --model {RUN_DIR}/mask.pt \
#     --scan /content/data/NFBS_Dataset/A00028185/sub-A00028185_ses-NFB3_T1w.nii.gz \
#     --out /content/A00028185_brainmask.nii.gz

# Evaluate against held-out reference masks (scans NOT used in training):
# !cd {CODE_DIR} && python evaluate.py \
#     --model {RUN_DIR}/mask.pt \
#     --data-dir /content/heldout \
#     --output-dir /content/evaluation --save-masks

# Copy the finished model off the runtime:
# !cp {RUN_DIR}/mask.pt {RUN_DIR}/mask.json {DRIVE_RUN_DIR}/

## 13. Evaluate a selected model across augmentation tiers

Run the setup/staging/dependency cells first; training is not required. Set `EVAL_MODEL` to any compatible checkpoint (local or Drive), then supply **one held-out manifest or directory**. Use separate inputs/output folders for validation and test. `DATA_ROOT` contains the whole training dataset and is deliberately not the default. Manifest records use `{"id": "subject_01", "scan": "/path/scan.nii.gz", "mask": "/path/reference.nii.gz"}`; JSON, JSONL, and CSV are supported.

Four cumulative tiers are reported:

1. **Clean:** original scans, without synthetic changes.
2. **Mild:** isolated rotation, resolution degradation, and anatomical resizing, plus their combination.
3. **Protocols:** mild conditions plus synthetic T2, MP2RAGE, FLAIR, MPRAGE, and other supported MRI contrasts.
4. **Full:** all preceding augmented conditions, supported individual 3D artifacts, paired morphology/label synthesis/appearance, and a mixed stress condition. This is a coverage suite; it does not apply every artifact simultaneously.

Each condition has its own Dice row. Clean scans are excluded from the augmented tier means. Repeats are averaged per condition, conditions per case, then cases equally; unchanged renders are excluded, and only cases with all planned trials successful contribute to a condition/tier mean. Coverage and failure counts remain visible. A separately labeled available-case Dice gives partial results when some draws fail or are unchanged; its case/condition mix can differ and it is not a complete-tier score. Synthetic contrast scores measure stress-test robustness, not performance on independently acquired T2/FLAIR/MP2RAGE datasets. Unsupported or specialized operations are listed with reasons, not silently counted as clean scans.

The same deployed preprocessing, threshold, and postprocessing are used for every trial. Geometry changes transform both scan and reference; resolution/appearance/artifacts retain the corresponding reference. Reference masks guide synthetic rendering only, never model normalization. Seeds use the resolved scan path and are stable across runs and manifest ordering. Keep staged scan paths fixed for paired model comparisons.

Start with `MAX_CASES = 5` for a pilot; set it to `None` for the complete held-out cohort. Increase `REPEATS` for more random draws. Full-volume synthesis and inference can take substantial time. CSV/JSON reports update after each condition directly in `EVAL_OUTPUT_DIR` (Drive by default); temporary volumes stay on local disk. Rerunning creates a fresh timestamped report folder. An interrupted or failed run is explicitly marked incomplete/failed.


In [ ]:
# Edit these settings, then run this cell. No training is launched.
from pathlib import Path
import html, json, os, signal, subprocess, sys, time
from IPython.display import HTML, display

EVAL_MODEL = f"{RUN_DIR}/mask.pt"  # or /content/drive/MyDrive/.../selected_model.pt
EVAL_MANIFEST = ""                # explicit held-out JSON/JSONL/CSV; takes precedence below
EVAL_DATA_DIR = "/content/heldout" # directory of held-out scan/reference pairs
EVAL_SCAN_GLOB = "*_T1w.nii.gz"
EVAL_MASK_SUFFIX = "_brainmask"    # e.g. _brainmask_refined_crf for matching labels
MODEL_RESULTS_JSON = ""           # only needed for a legacy checkpoint without embedded settings

TIERS = ["clean", "mild", "protocols", "full"]
MAX_CASES = 5                     # None = all supplied held-out cases; 5 = reproducible pilot subset
REPEATS = 1                       # random draws per augmented condition; clean runs once
SEED = 2026                      # keep fixed when comparing checkpoints
THRESHOLD = 0.60                  # keep the same operating point across all tiers/models
ROTATION_DEGREES = 10.0
RESIZE_RANGE = (0.90, 1.10)        # anatomical scale; paired scan + reference transform
RESOLUTION_RANGE = (1.20, 1.80)    # resolution degradation factors; reference unchanged
ARTIFACT_SEVERITY = (0.15, 0.45)   # per-operator severity scale, not a shared physical unit
EVAL_OUTPUT_DIR = f"{DRIVE_RUN_DIR}/augmentation_eval_{time.strftime('%Y%m%d_%H%M%S')}_{time.time_ns() % 1000000:06d}"

assert not Path(EVAL_OUTPUT_DIR, "results.json").exists(), "Choose a fresh evaluation output folder."
assert Path(EVAL_MODEL).is_file(), f"Checkpoint not found: {EVAL_MODEL}"
_eval_required = (
    'generate_mask.py',
    'evaluate.py',
    'evaluate_augmentations.py',
    'inference/__init__.py',
    'inference/masker.py',
    'inference/checkpoints.py',
    'evaluation/__init__.py',
    'evaluation/standard.py',
    'evaluation/augmentation.py',
    'evaluation/transforms.py',
    'configuration/run_config.augmented.json',
    'imaging/__init__.py',
    'imaging/geometry.py', 'imaging/normalization.py', 'imaging/metadata.py',
)
_eval_missing = [name for name in _eval_required if not Path(CODE_DIR, name).is_file()]
assert not _eval_missing, ("Augmentation evaluation needs these files in the bundle: " +
                           ", ".join(_eval_missing) + ". Select the updated bundle in CODE_SRC and re-stage it.")
_source = ["--manifest", EVAL_MANIFEST] if EVAL_MANIFEST else ["--data-dir", EVAL_DATA_DIR]
assert Path(_source[1]).exists(), f"Supply an explicit held-out manifest/directory: {_source[1]}"
_cmd = [sys.executable, "-u", "evaluate_augmentations.py", "--model", EVAL_MODEL,
        *_source, "--output-dir", EVAL_OUTPUT_DIR, "--scan-glob", EVAL_SCAN_GLOB,
        "--mask-suffix", EVAL_MASK_SUFFIX, "--tiers", *TIERS,
        "--repeats", str(REPEATS), "--seed", str(SEED), "--threshold", str(THRESHOLD),
        "--rotation-degrees", str(ROTATION_DEGREES),
        "--resize-range", *map(str, RESIZE_RANGE),
        "--resolution-range", *map(str, RESOLUTION_RANGE),
        "--artifact-severity", *map(str, ARTIFACT_SEVERITY)]
if MAX_CASES is not None:
    _cmd += ["--max-cases", str(MAX_CASES)]
if MODEL_RESULTS_JSON:
    _cmd += ["--config", MODEL_RESULTS_JSON]

# A fresh subprocess avoids cached model/augmentation imports from previous notebook runs.
_env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")
print("Pilot case limit:" if MAX_CASES is not None else "Full supplied cohort:", MAX_CASES or "all")
print("Reports:", EVAL_OUTPUT_DIR)
_proc = subprocess.Popen(_cmd, cwd=CODE_DIR, env=_env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for _line in _proc.stdout:
        print(_line, end="")
    _rc = _proc.wait()
finally:
    if _proc.poll() is None:
        _proc.send_signal(signal.SIGINT)  # let the evaluator publish interrupted status
        try:
            _proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            _proc.kill()
            _proc.wait()

_report_path = Path(EVAL_OUTPUT_DIR, "results.json")
if not _report_path.is_file():
    raise RuntimeError("Evaluation did not start; inspect the error above.")
_report = json.loads(_report_path.read_text(encoding="utf-8"))

def _show_scores(title, rows, key):
    headers = ["Condition", "Complete-case Dice", "Available-case Dice*", "Median", "Worst case mean", "Complete cases", "Trials OK / planned", "Failed", "Unchanged", "Pending"]
    body = []
    for row in rows:
        pct = lambda name: "—" if row[name] is None else f"{100 * row[name]:.3f}%"
        vals = [row[key], pct("mean_dice"), pct("available_mean_dice"), pct("median_dice"), pct("min_dice"),
                f"{row['complete_cases']} / {row['total_cases']}",
                f"{row['successful_trials']} / {row['expected_trials']}", row["failed_trials"], row["unchanged_trials"], row["pending_trials"]]
        body.append("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in vals) + "</tr>")
    display(HTML(f"<h3>{html.escape(title)}</h3><table><thead><tr>" +
                 "".join(f"<th>{h}</th>" for h in headers) + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))

print("Status:", _report["status"], "| evaluated cases:", _report["case_count"])
_show_scores("Tier Dice (cumulative augmented conditions)", _report["tiers"], "label")
for _tier in _report["tiers"]:
    _names = set(_tier["augmentations"])
    _show_scores(f"{_tier['label']}: individual conditions",
                 [r for r in _report["augmentations"] if r["augmentation"] in _names], "augmentation")
_excluded = _report["catalog"].get("excluded", [])
if _excluded:
    display(HTML("<details><summary>Excluded / covered elsewhere operations</summary><ul>" +
                 "".join(f"<li>{html.escape(r['name'])}: {html.escape(r['reason'])}</li>" for r in _excluded) + "</ul></details>"))
print("* Available-case Dice excludes failed/unchanged draws, but coverage may differ: use complete-case Dice for fully matched comparisons.")
print("Saved: results.json (settings + seeds + per-trial details), cases.csv, summary.csv")
if _rc:
    raise RuntimeError("Benchmark contains failed trials. The tables show coverage; inspect failed rows in cases.csv/results.json.")
